# PhoBERT v2: cleaned data, with and without ViHSD

Setup: **GPU T4 x2**, **Internet On**, secret `HF_TOKEN`.
**Input:** the same private dataset as v1 (the three `vfyr_*_cleaned.csv` files). Nothing else to upload:
the notebook rebuilds the cleaned splits itself.

1. `find_label_issues.py` – confident learning flags likely label errors (5-fold CV + cleanlab)
2. `make_clean_splits.py --prune` → `data/clean/`: leakage removed, flagged rows pruned from train only
3. `add_vihsd.py` → `data/clean_vihsd/`: + 22.5k UIT-ViHSD comments in train; `data/vihsd_test/`: out-of-domain benchmark

Runs C (clean) and D (clean + ViHSD) share the same val/test, so they are directly comparable.
Both use **no class weights** and up to 6 epochs with early stopping.

In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji cleanlab
import glob, os, shutil, json
os.makedirs("data/raw", exist_ok=True)
for split in ["train", "val", "test"]:
    matches = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if split in os.path.basename(p).lower()]
    assert len(matches) == 1, f"Expected exactly one CSV for '{split}', found: {matches}"
    shutil.copy(matches[0], f"data/raw/{split}.csv")
    print(split, "<-", matches[0])
os.makedirs("reports", exist_ok=True)
!python scripts/find_label_issues.py | tail -5
!python scripts/make_clean_splits.py --prune | grep -A4 '"final"'
!python scripts/add_vihsd.py

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
# Run C: our cleaned data only
!python scripts/train_phobert.py --data-dir data/clean --no-class-weights --epochs 6 --output-dir models/C --tag _clean_nocw --no-push

In [ ]:
# Run D: our cleaned data + ViHSD train/dev
!python scripts/train_phobert.py --data-dir data/clean_vihsd --no-class-weights --epochs 6 --output-dir models/D --tag _clean_vihsd_nocw --no-push

In [ ]:
# Out-of-domain benchmark: ViHSD test
!python scripts/evaluate_model.py --model models/C/best --data-dir data/vihsd_test --name phobert_C --tag _vihsd_test
!python scripts/evaluate_model.py --model models/D/best --data-dir data/vihsd_test --name phobert_D --tag _vihsd_test

In [ ]:
c = json.load(open("reports/phobert_results_clean_nocw.json"))
d = json.load(open("reports/phobert_results_clean_vihsd_nocw.json"))
c["vihsd_test"] = json.load(open("reports/phobert_C_eval_vihsd_test.json"))
d["vihsd_test"] = json.load(open("reports/phobert_D_eval_vihsd_test.json"))
for name, r in [("C clean", c), ("D clean+ViHSD", d)]:
    print(f"{name}: val F1={r['val_f1_macro']} | test F1={r['test']['f1_macro']} "
          f"off={r['test']['f1_per_class']['offensive']} | ViHSD test F1={r['vihsd_test']['f1_macro']}")
# Pick on validation only. D is preferred unless clearly worse (it generalizes to other sources).
best = "models/D/best" if d["val_f1_macro"] >= c["val_f1_macro"] - 0.005 else "models/C/best"
print("Pushing", best)
from huggingface_hub import HfApi
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
HfApi().upload_folder(folder_path=best, repo_id=repo_id, commit_message=f"v2: trained on {best}")
print(f"https://huggingface.co/{repo_id}")

Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps({"C_clean_nocw": c, "D_clean_vihsd_nocw": d}, ensure_ascii=False))
